## FIFA PREDICTOR

Predicts the outcome of a match between any two national teams by combining three signals:

1. **Match history (1930-2022 World Cups)** &rarr; an **Elo rating** for each team that evolves match-by-match.
2. **Current FIFA ranking** (points/rank) &rarr; an up-to-date snapshot of official strength, useful for teams whose Elo is stale or thin on history.
3. **Player-level squad strength** (`CLEAN_FIFA23_official_data.csv`) &rarr; average `Overall` rating of each nation's best 23 players, broken down by GK/DEF/MID/FWD.

These are blended into a single **strength score** per team, fed into a logistic regression to get Win/Draw/Loss probabilities, and combined with a **Poisson goal-expectancy model** (built from historical World Cup goals) to simulate a likely scoreline.

### 1. Load the data

In [1]:
import pandas as pd
import numpy as np

world_cup = pd.read_csv('1930-2022/world_cup.csv')          # basic results of all World Cups 1930-2022
ranks = pd.read_csv('1930-2022/fifa_ranking_2026-06-08.csv') # current FIFA ranks & points for all teams
clean_data = pd.read_csv('CLEAN_DATA/CLEAN_FIFA23_official_data.csv') # player-level data, last FIFA edition
matches = pd.read_csv('1930-2022/matches_1930_2022.csv')     # all World Cup match-level statistics

matches['Date'] = pd.to_datetime(matches['Date'])
matches = matches.sort_values('Date').reset_index(drop=True)

world_cup

,Year,Host,Teams,Champion,Runner-Up,TopScorrer,Attendance,AttendanceAvg,Matches
0,2022,Qatar,32,Argentina,France,Kylian Mbappé - 8,3404252,53191,64
1,2018,Russia,32,France,Croatia,Harry Kane - 6,3031768,47371,64
2,2014,Brazil,32,Germany,Argentina,James Rodríguez - 6,3429873,53592,64
3,2010,South Africa,32,Spain,Netherlands,"Wesley Sneijder, Thomas Müller... - 5",3178856,49670,64
4,2006,Germany,32,Italy,France,Miroslav Klose - 5,3352605,52384,64
5,2002,"Korea Republic, Japan",32,Brazil,Germany,Ronaldo - 8,2705337,42271,64
6,1998,France,32,France,Brazil,Davor Šuker - 6,2903477,45367,64
7,1994,United States,24,Brazil,Italy,"Hristo Stoichkov, Oleg Salenko - 6",3587538,68991,52
8,1990,Italy,24,West Germany,Argentina,Salvatore Schillaci - 6,2516215,48389,52
9,1986,Mexico,24,Argentina,West Germany,Gary Lineker - 6,2394031,46039,52


In [2]:
ranks.head()

,team,team_code,association,rank,previous_rank,points,previous_points,rated_matches
0,Argentina,ARG,CONMEBOL,1,3,1876.118331,1874.814835,59
1,Spain,ESP,UEFA,2,2,1873.013187,1876.395199,56
2,France,FRA,UEFA,3,1,1869.428449,1877.322731,57
3,England,ENG,UEFA,4,4,1827.048678,1825.965482,57
4,Portugal,POR,UEFA,5,5,1766.177547,1763.834406,56


In [3]:
clean_data.head()

,Unnamed: 0,ID,Name,Age,Photo,Nationality,Flag,Overall,Potential,Club,...,Position,Joined,Loaned From,Contract Valid Until,Height(cm.),Weight(lbs.),Release Clause(£),Kit Number,Best Overall Rating,Year_Joined
0,0,209658,L. Goretzka,27,https://cdn.sofifa.net/players/209/658/23_60.png,Germany,https://cdn.sofifa.net/flags/de.png,87,88,FC Bayern München,...,SUB,2018-07-01,NaN,2026.0,189.0,180.810,157000000.0,8.0,0.0,2018
1,1,212198,Bruno Fernandes,27,https://cdn.sofifa.net/players/212/198/23_60.png,Portugal,https://cdn.sofifa.net/flags/pt.png,86,87,Manchester United,...,LCM,2020-01-30,NaN,2026.0,179.0,152.145,155000000.0,8.0,0.0,2020
2,2,224334,M. Acuña,30,https://cdn.sofifa.net/players/224/334/23_60.png,Argentina,https://cdn.sofifa.net/flags/ar.png,85,85,Sevilla FC,...,LB,2020-09-14,NaN,2024.0,172.0,152.145,97700000.0,19.0,0.0,2020
3,3,192985,K. De Bruyne,31,https://cdn.sofifa.net/players/192/985/23_60.png,Belgium,https://cdn.sofifa.net/flags/be.png,91,91,Manchester City,...,RCM,2015-08-30,NaN,2025.0,181.0,154.350,198900000.0,17.0,0.0,2015
4,4,224232,N. Barella,25,https://cdn.sofifa.net/players/224/232/23_60.png,Italy,https://cdn.sofifa.net/flags/it.png,86,89,Inter,...,RCM,2020-09-01,NaN,2026.0,172.0,149.940,154400000.0,23.0,0.0,2020


In [4]:
matches.head()

,home_team,away_team,home_score,home_xg,home_penalty,away_score,away_xg,away_penalty,home_manager,home_captain,...,home_penalty_shootout_miss_long,away_penalty_shootout_miss_long,home_red_card,away_red_card,home_yellow_red_card,away_yellow_red_card,home_yellow_card_long,away_yellow_card_long,home_substitute_in_long,away_substitute_in_long
0,France,Mexico,4,NaN,NaN,1,NaN,NaN,Raoul Caudron,Alexandre Villaplane,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United States,Belgium,3,NaN,NaN,0,NaN,NaN,Bob Millar,Tom Florie,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Yugoslavia,Brazil,2,NaN,NaN,1,NaN,NaN,Bosko Simonovic,Milutin Ivković,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Romania,Peru,3,NaN,NaN,1,NaN,NaN,Octav Luchide,Emerich Vogl,...,NaN,NaN,NaN,Plácido Galindo · 70,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,France,1,NaN,NaN,0,NaN,NaN,Francisco Olazar,Manuel Ferreira,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. Reconcile team names across the three datasets

The match history contains retired/historical names (`West Germany`, `Czechoslovakia`, `Soviet Union`, ...) and the player and ranking files spell a few countries differently (`IR Iran` vs `Iran`, `Cabo Verde` vs `Cape Verde Islands`, `Curaçao` vs `Curacao`, etc.). We map everything onto one canonical name so a team's history, rank, and squad all line up.

In [5]:
NAME_MAP = {
    # historical teams -> present-day successor
    'West Germany': 'Germany', 'Germany DR': 'Germany',
    'Czechoslovakia': 'Czechia', 'Czech Republic': 'Czechia',
    'Soviet Union': 'Russia',
    'Yugoslavia': 'Serbia', 'FR Yugoslavia': 'Serbia', 'Serbia and Montenegro': 'Serbia',
    'Zaire': 'DR Congo',
    'Dutch East Indies': 'Indonesia',
    # spelling differences between files
    'IR Iran': 'Iran',
    'Hong Kong, China': 'Hong Kong',
    'Cabo Verde': 'Cape Verde Islands',
    'Guinea-Bissau': 'Guinea Bissau',
    'Türkiye': 'Turkey',
    'Curaçao': 'Curacao',
    'United States': 'USA',
}

def norm(name):
    return NAME_MAP.get(name, name)

matches['home_team'] = matches['home_team'].apply(norm)
matches['away_team'] = matches['away_team'].apply(norm)
ranks['team'] = ranks['team'].apply(norm)
clean_data['Nationality'] = clean_data['Nationality'].apply(norm)

### 3. Build an Elo rating from match history

A standard football-Elo update: each match moves both teams' ratings based on the gap between the *expected* result (from their pre-match ratings) and the *actual* result, scaled up for bigger score margins. This turns 84 years of match outcomes into one number per team that reflects sustained World Cup performance.

In [6]:
BASE_ELO = 1500
K = 30
HOME_ADV = 60  # elo-point bonus for the nominal home / host team

elo = {}
def get_elo(team):
    return elo.get(team, BASE_ELO)

elo_rows = []
for _, m in matches.iterrows():
    h, a = m['home_team'], m['away_team']
    hs, as_ = m['home_score'], m['away_score']
    if pd.isna(hs) or pd.isna(as_):
        continue

    elo_h, elo_a = get_elo(h), get_elo(a)
    elo_rows.append({'home_team': h, 'away_team': a, 'Date': m['Date'],
                      'elo_home': elo_h, 'elo_away': elo_a,
                      'home_score': hs, 'away_score': as_})

    exp_h = 1 / (1 + 10 ** (-((elo_h + HOME_ADV) - elo_a) / 400))
    res_h = 1.0 if hs > as_ else (0.5 if hs == as_ else 0.0)

    gd = abs(hs - as_)
    mult = 1 if gd <= 1 else (1.5 if gd == 2 else (1.75 + (gd - 3) / 8))

    delta = K * mult * (res_h - exp_h)
    elo[h] = elo_h + delta
    elo[a] = elo_a - delta

elo_df = pd.DataFrame(elo_rows)
pd.Series(elo).sort_values(ascending=False).head(15)

Netherlands    1746.104237
France         1734.781633
Germany        1723.540459
Brazil         1699.878270
Argentina      1673.008388
Italy          1638.609499
England        1626.510966
Spain          1617.471942
Croatia        1603.024035
Belgium        1594.871131
Portugal       1576.286474
Sweden         1568.771147
Colombia       1560.719540
Turkey         1546.677906
Uruguay        1544.370853
dtype: float64

### 4. Squad strength from player-level data

For each nationality, take the top 23 players by `Overall` (a reasonable stand-in for a World Cup squad) and average their rating, both overall and by position group.

In [7]:
def position_group(pos):
    if pos == 'GK':
        return 'GK'
    if pos in ('SUB', 'RES'):
        return 'OTHER'
    if pos == 'CB' or ('B' in pos and pos not in ('CB',)) or pos.endswith('B'):
        return 'DEF'
    if 'M' in pos:
        return 'MID'
    return 'FWD'

clean_data['pos_group'] = clean_data['Position'].apply(position_group)

SQUAD_SIZE = 23
def squad_strength(team):
    squad = clean_data[clean_data['Nationality'] == team].sort_values('Overall', ascending=False).head(SQUAD_SIZE)
    if len(squad) == 0:
        return None
    out = {'overall': squad['Overall'].mean(), 'potential': squad['Potential'].mean(), 'n_players': len(squad)}
    for grp in ['GK', 'DEF', 'MID', 'FWD']:
        sub = squad[squad['pos_group'] == grp]
        out[f'{grp.lower()}_overall'] = sub['Overall'].mean() if len(sub) else np.nan
    return out

squad_cache = {}
def get_squad(team):
    if team not in squad_cache:
        squad_cache[team] = squad_strength(team)
    return squad_cache[team]

get_squad('Argentina'), get_squad('France')

({'overall': np.float64(82.91304347826087),
  'potential': np.float64(84.04347826086956),
  'n_players': 23,
  'gk_overall': np.float64(82.33333333333333),
  'def_overall': np.float64(82.5),
  'mid_overall': np.float64(81.0),
  'fwd_overall': np.float64(85.16666666666667)},
 {'overall': np.float64(85.0),
  'potential': np.float64(86.65217391304348),
  'n_players': 23,
  'gk_overall': np.float64(87.0),
  'def_overall': np.float64(83.57142857142857),
  'mid_overall': np.float64(86.0),
  'fwd_overall': np.float64(85.66666666666667)})

### 5. Current FIFA ranking lookup

In [8]:
rank_lookup = ranks.set_index('team')[['rank', 'points']].to_dict('index')

def get_rank(team):
    return rank_lookup.get(team, {'rank': np.nan, 'points': np.nan})

get_rank('Argentina'), get_rank('France')

({'rank': 1, 'points': 1876.118331}, {'rank': 3, 'points': 1869.428449})

### 6. Train an outcome model on Elo difference

Target: 0 = away win, 1 = draw, 2 = home win. A simple logistic regression on `elo_diff` already captures most of the signal in football outcomes — draws are inherently the hardest class to call (true of almost every football prediction model, not just this one).

In [10]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 11.1 MB 5.0 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 19.6 MB/s eta 0:00:01     |██████████████████▍             | 17.5 MB 8.8 MB/s eta 0:00:02     |██████████████████████████▍     | 25.0 MB 8.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

elo_df['elo_diff'] = elo_df['elo_home'] - elo_df['elo_away']

def outcome(row):
    if row['home_score'] > row['away_score']:
        return 2
    elif row['home_score'] == row['away_score']:
        return 1
    return 0

elo_df['target'] = elo_df.apply(outcome, axis=1)

X = elo_df[['elo_diff']].values
y = elo_df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=['Away win', 'Draw', 'Home win'], zero_division=0))

Accuracy: 0.5854922279792746
              precision    recall  f1-score   support

    Away win       0.37      0.17      0.24        40
        Draw       0.00      0.00      0.00        43
    Home win       0.61      0.96      0.75       110

    accuracy                           0.59       193
   macro avg       0.33      0.38      0.33       193
weighted avg       0.42      0.59      0.47       193



### 7. Blend match-history Elo with current FIFA rank + squad strength

Elo is built only from World Cup matches, so a team with little World Cup history (or one that has improved/declined a lot since its last appearance) gets a stale rating. FIFA points and current squad `Overall` are available for **every** team, including ones with sparse World Cup history. We fit a small linear regression on the teams where we have all three signals (`elo ~ fifa_points + squad_overall`), then use that regression to estimate a strength score for any team &mdash; blending it with actual Elo when history exists.

In [12]:
from sklearn.linear_model import LinearRegression

rows = []
for team, e in elo.items():
    rk = get_rank(team)
    sq = get_squad(team)
    if pd.isna(rk['points']) or sq is None:
        continue
    rows.append({'team': team, 'elo': e, 'points': rk['points'], 'squad_overall': sq['overall']})

bridge_df = pd.DataFrame(rows)
bridge_X = bridge_df[['points', 'squad_overall']].values
bridge_y = bridge_df['elo'].values
bridge_model = LinearRegression().fit(bridge_X, bridge_y)

print(f"Teams used to fit bridge model: {len(bridge_df)}")
print("Bridge R^2:", bridge_model.score(bridge_X, bridge_y))

def estimated_strength(team):
    """Blended strength score for ANY team, in Elo-equivalent units."""
    has_elo = team in elo
    rk = get_rank(team)
    sq = get_squad(team)
    has_bridge = not pd.isna(rk.get('points', np.nan)) and sq is not None

    bridge_est = None
    if has_bridge:
        bridge_est = bridge_model.predict([[rk['points'], sq['overall']]])[0]

    if has_elo and bridge_est is not None:
        return 0.6 * elo[team] + 0.4 * bridge_est   # weight real history higher, but pull toward current form
    elif has_elo:
        return elo[team]
    elif bridge_est is not None:
        return bridge_est
    else:
        return BASE_ELO  # no data at all -> neutral

Teams used to fit bridge model: 78
Bridge R^2: 0.5575871031909498


### 8. Expected goals (Poisson attack/defense model)

Classic football-analytics approach: each team gets an *attack factor* (their average goals scored, relative to the tournament average) and a *defense factor* (average goals conceded, relative to average). Expected goals for team A vs team B = tournament-average goals &times; A's attack factor &times; B's defense factor, nudged slightly by the overall strength gap so match-ups with little head-to-head history still get a sensible scoreline.

In [13]:
avg_goals = pd.concat([matches['home_score'], matches['away_score']]).mean()

attack, defense, games = {}, {}, {}
for _, m in matches.iterrows():
    if pd.isna(m['home_score']) or pd.isna(m['away_score']):
        continue
    for team, gf, ga in [(m['home_team'], m['home_score'], m['away_score']),
                          (m['away_team'], m['away_score'], m['home_score'])]:
        attack[team] = attack.get(team, 0) + gf
        defense[team] = defense.get(team, 0) + ga
        games[team] = games.get(team, 0) + 1

def attack_factor(team):
    if team not in games or games[team] == 0:
        return 1.0
    return (attack[team] / games[team]) / avg_goals

def defense_factor(team):
    if team not in games or games[team] == 0:
        return 1.0
    return (defense[team] / games[team]) / avg_goals

def expected_goals(team_a, team_b, elo_diff):
    base = avg_goals * attack_factor(team_a) * defense_factor(team_b)
    adj = np.clip(elo_diff / 400, -0.75, 0.75)
    return max(0.15, base + adj * 0.5)

### 9. `predict_match()` &mdash; the final predictor

Combines everything: blended strength &rarr; Elo-model Win/Draw/Loss probabilities, plus a Monte-Carlo simulation of the Poisson goal model for a most-likely scoreline and a second, goal-based set of outcome probabilities to sanity-check the first.

In [30]:
def predict_match(team_a, team_b, neutral=True, sims=20000, seed=0):
    team_a, team_b = norm(team_a), norm(team_b)
    strength_a = estimated_strength(team_a)
    strength_b = estimated_strength(team_b)
    elo_diff = (strength_a - strength_b) + (0 if neutral else HOME_ADV)

    probs = clf.predict_proba([[elo_diff]])[0]  # [away_win, draw, home_win] -> [B win, draw, A win]
    p_b, p_draw, p_a = probs

    lam_a = expected_goals(team_a, team_b, elo_diff)
    lam_b = expected_goals(team_b, team_a, -elo_diff)

    rng = np.random.default_rng(seed)
    goals_a = rng.poisson(lam_a, sims)
    goals_b = rng.poisson(lam_b, sims)
    sim_a_win = (goals_a > goals_b).mean()
    sim_draw = (goals_a == goals_b).mean()
    sim_b_win = (goals_a < goals_b).mean()

    return {
        'team_a': team_a, 'team_b': team_b,
        'elo_model_probs': {'team_a_win': round(float(p_a), 3), 'draw': round(float(p_draw), 3), 'team_b_win': round(float(p_b), 3)},
        #'poisson_sim_probs': {'team_a_win': round(float(sim_a_win), 3), 'draw': round(float(sim_draw), 3), 'team_b_win': round(float(sim_b_win), 3)},
       # 'expected_score': f"{team_a} {lam_a:.1f} - {lam_b:.1f} {team_b}",
        'strength_a': round(float(strength_a), 1), 'strength_b': round(float(strength_b), 1),
        'fifa_rank_a': get_rank(team_a).get('rank'), 'fifa_rank_b': get_rank(team_b).get('rank'),
    }

predict_match('Argentina', 'France')

{'team_a': 'Argentina',
 'team_b': 'France',
 'elo_model_probs': {'team_a_win': 0.485, 'draw': 0.218, 'team_b_win': 0.297},
 'strength_a': 1641.5,
 'strength_b': 1685.2,
 'fifa_rank_a': 1,
 'fifa_rank_b': 3}

In [33]:
predict_match('Brazil', 'Japan')

{'team_a': 'Brazil',
 'team_b': 'Japan',
 'elo_model_probs': {'team_a_win': 0.656, 'draw': 0.232, 'team_b_win': 0.112},
 'strength_a': 1661.8,
 'strength_b': 1496.8,
 'fifa_rank_a': 6,
 'fifa_rank_b': 18}

In [31]:
predict_match('Spain', 'France', neutral=False)  

{'team_a': 'Spain',
 'team_b': 'France',
 'elo_model_probs': {'team_a_win': 0.516, 'draw': 0.224, 'team_b_win': 0.26},
 'strength_a': 1613.5,
 'strength_b': 1685.2,
 'fifa_rank_a': 2,
 'fifa_rank_b': 3}

In [32]:
predict_match('Argentina', 'England', neutral=True)  

{'team_a': 'Argentina',
 'team_b': 'England',
 'elo_model_probs': {'team_a_win': 0.55, 'draw': 0.229, 'team_b_win': 0.221},
 'strength_a': 1641.5,
 'strength_b': 1616.4,
 'fifa_rank_a': 1,
 'fifa_rank_b': 4}

In [34]:
predict_match('Argentina','Spain', neutral = True)

{'team_a': 'Argentina',
 'team_b': 'Spain',
 'elo_model_probs': {'team_a_win': 0.553, 'draw': 0.229, 'team_b_win': 0.218},
 'strength_a': 1641.5,
 'strength_b': 1613.5,
 'fifa_rank_a': 1,
 'fifa_rank_b': 2}